# Model Fine-Tuning

Main 100-epoch LoRA training notebook. Reusable logic lives in `src/`.

In [ ]:
import yaml

dataset_cfg = yaml.safe_load(open('configs/dataset.yaml', encoding='utf-8'))
model_cfg = yaml.safe_load(open('configs/model.yaml', encoding='utf-8'))
train_cfg = yaml.safe_load(open('configs/training.yaml', encoding='utf-8'))
train_cfg['num_train_epochs']

In [ ]:
from src.data.dataset_loader import AudioQADataset, load_split
from src.data.collator import Qwen2AudioQACollator
from src.models.qwen2_audio import load_qwen2_audio
from src.models.lora import apply_lora, parameter_report
from src.training.trainer import build_training_args, build_trainer
from src.utils.seed import set_seed

set_seed(train_cfg['seed'])
train_records = load_split('train')
val_records = load_split('validation')
len(train_records), len(val_records)

In [ ]:
model, processor = load_qwen2_audio(model_cfg['model_name'], **model_cfg.get('from_pretrained_kwargs', {}))
model = apply_lora(model, train_cfg['lora'])
parameter_report(model)

In [ ]:
collator = Qwen2AudioQACollator(processor, sampling_rate=model_cfg['sampling_rate'])
args = build_training_args(train_cfg)
trainer = build_trainer(model, args, AudioQADataset(train_records), AudioQADataset(val_records), collator, train_cfg)
trainer.train()
trainer.save_model(train_cfg['best_model_dir'])
processor.save_pretrained(train_cfg['best_model_dir'])

In [ ]:
from src.training.plots import plot_training_curves

plot_training_curves(train_cfg['log_file'], 'results/plots')